In [1]:
import pandas as pd
import numpy as np

# ── Load ──
micro = pd.read_csv("micro_clean.csv")
large = pd.read_csv("large_clean.csv")

# ── Parse hashtags ──
def parse_micro_hashtags(row):
    t = str(row['hashtags_from_title']).split('|||')
    d = str(row['hashtags_from_desc']).split('|||')
    return list(set([x.strip().lower().lstrip('#') for x in t+d if x.strip() and x.strip()!='nan']))

def parse_large_hashtags(x):
    x = str(x)
    if x in ['nan','[]','']: return []
    x = x.strip("[]").replace("'","").replace('"','')
    return list(set([t.strip().lower() for t in x.split(',') if t.strip()]))

micro['hashtag_list'] = micro.apply(parse_micro_hashtags, axis=1)
large['hashtag_list'] = large['all_hashtags'].apply(parse_large_hashtags)

# ── Combine ──
shared = ['video_id','view_count','like_rate','subscriber_count',
          'category_name','source','hashtag_count',
          'publish_hour','duration_sec','hashtag_list']

combined = pd.concat([micro[shared], large[shared]], ignore_index=True)

print("Combined shape:", combined.shape)
print("\nCategory distribution:")
cat_dist = combined.groupby(['category_name','source']).size().unstack(fill_value=0)
cat_dist['total'] = cat_dist.sum(axis=1)
print(cat_dist.sort_values('total', ascending=False).to_string())

Combined shape: (2212, 10)

Category distribution:
source                 large  micro  total
category_name                             
People & Blogs           425    766   1191
Entertainment            209     82    291
Gaming                   117     48    165
Sports                   101      7    108
Film & Animation          70     14     84
Autos & Vehicles          72      3     75
Education                 17     45     62
Comedy                    42     12     54
Music                     42     11     53
Howto & Style             20     24     44
Pets & Animals            34      2     36
Travel & Events           14     11     25
Other                     14      0     14
Science & Technology       0      9      9
Nonprofits & Activism      0      1      1


In [2]:
import pandas as pd
import numpy as np

# ── Load combined data (already done above) ──
# Focus on categories with 50+ videos
valid_cats = ['People & Blogs', 'Entertainment', 'Gaming', 
              'Sports', 'Film & Animation', 'Autos & Vehicles',
              'Education', 'Comedy', 'Music']

combined_valid = combined[combined['category_name'].isin(valid_cats)].copy()

# ── Four Quadrant Analysis per category ──
results = []

for cat in valid_cats:
    subset = combined_valid[combined_valid['category_name'] == cat].copy()
    
    # Use category-level median as threshold
    med_views = subset['view_count'].median()
    med_lr    = subset['like_rate'].median()
    
    # Assign quadrant
    def get_quadrant(row):
        high_v = row['view_count'] >= med_views
        high_l = row['like_rate']  >= med_lr
        if high_v and high_l:   return 'Golden'
        elif high_v and not high_l: return 'Exposure Trap'
        elif not high_v and high_l: return 'Niche'
        else: return 'Ineffective'
    
    subset['quadrant'] = subset.apply(get_quadrant, axis=1)
    
    # Top tags in Golden quadrant
    golden = subset[subset['quadrant'] == 'Golden']
    tag_perf = {}
    for _, row in golden.iterrows():
        for t in row['hashtag_list']:
            tag_perf.setdefault(t, []).append(row['like_rate'])
    
    top_golden_tags = sorted(
        [(t, len(v), round(np.mean(v), 4)) for t, v in tag_perf.items() if len(v) >= 3],
        key=lambda x: -x[1]
    )[:5]
    
    # Quadrant distribution
    quad_dist = subset['quadrant'].value_counts()
    
    print(f"\n[{cat}] n={len(subset)}, med_views={int(med_views)}, med_lr={round(med_lr,4)}")
    print(f"  Quadrant distribution: {dict(quad_dist)}")
    print(f"  Top Golden tags: {[(t[0], t[1]) for t in top_golden_tags]}")


[People & Blogs] n=1191, med_views=1420, med_lr=0.0193
  Quadrant distribution: {'Golden': np.int64(335), 'Ineffective': np.int64(334), 'Niche': np.int64(261), 'Exposure Trap': np.int64(261)}
  Top Golden tags: [('shorts', 79), ('vlog', 21), ('viral', 16), ('funny', 13), ('trending', 11)]

[Entertainment] n=291, med_views=1791373, med_lr=0.021
  Quadrant distribution: {'Niche': np.int64(78), 'Exposure Trap': np.int64(78), 'Golden': np.int64(68), 'Ineffective': np.int64(67)}
  Top Golden tags: [('shorts', 21), ('viral', 8), ('comedy', 7), ('edit', 7), ('fyp', 6)]

[Gaming] n=165, med_views=246121, med_lr=0.0265
  Quadrant distribution: {'Golden': np.int64(53), 'Ineffective': np.int64(52), 'Niche': np.int64(30), 'Exposure Trap': np.int64(30)}
  Top Golden tags: [('roblox', 4), ('minecraft', 4), ('shorts', 3), ('residentevilrequiem', 3)]

[Sports] n=108, med_views=1917024, med_lr=0.0189
  Quadrant distribution: {'Niche': np.int64(31), 'Exposure Trap': np.int64(31), 'Ineffective': np.int6

In [4]:
import numpy as np
from scipy import stats

# Use micro dataset
df_reg = micro[['hashtag_count', 'like_rate', 'view_count', 
                'duration_sec', 'subscriber_count']].dropna()

# Simple correlation test with p-value
corr, p_value = stats.pearsonr(df_reg['hashtag_count'], df_reg['like_rate'])
print(f"Pearson correlation: {round(corr, 4)}, p-value: {round(p_value, 4)}")

# Spearman (more robust, doesn't assume normal distribution)
corr_s, p_value_s = stats.spearmanr(df_reg['hashtag_count'], df_reg['like_rate'])
print(f"Spearman correlation: {round(corr_s, 4)}, p-value: {round(p_value_s, 4)}")

# Partial correlation: control for view_count and subscriber_count
from numpy.linalg import lstsq

def partial_corr(x, y, controls):
    # Residualize x and y on controls
    controls_arr = np.column_stack([controls, np.ones(len(controls))])
    x_resid = x - controls_arr @ lstsq(controls_arr, x, rcond=None)[0]
    y_resid = y - controls_arr @ lstsq(controls_arr, y, rcond=None)[0]
    corr_p, p_val_p = stats.pearsonr(x_resid, y_resid)
    return corr_p, p_val_p

controls = df_reg[['view_count', 'subscriber_count', 'duration_sec']].values
corr_p, p_val_p = partial_corr(
    df_reg['hashtag_count'].values,
    df_reg['like_rate'].values,
    controls
)
print(f"\nPartial correlation (controlling for views, subscribers, duration):")
print(f"  correlation: {round(corr_p, 4)}, p-value: {round(p_val_p, 4)}")
print(f"  {'NOT significant (p>0.05) — hashtag count has no significant effect' if p_val_p > 0.05 else 'Significant (p<0.05) — hashtag count has some effect'}")

Pearson correlation: -0.0234, p-value: 0.4521
Spearman correlation: -0.118, p-value: 0.0001

Partial correlation (controlling for views, subscribers, duration):
  correlation: -0.0229, p-value: 0.4612
  NOT significant (p>0.05) — hashtag count has no significant effect


In [4]:
# ============================================================
# Regenerate all charts with template colors
# Primary: #BA3A2C (red), #222222 (black)
# ============================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from wordcloud import WordCloud

RED    = "#BA3A2C"
BLACK  = "#222222"
GRAY   = "#888888"
LGRAY  = "#DDDDDD"
BGRAY  = "#F4F4F4"

# ── 1. Word Cloud ──
top15_lr_freq = {
    'simplelife': 1577, 'firstvideo': 1513, 'youtubegrowth': 1290,
    'smallcreator': 1153, 'gaming': 1114, 'dailyroutine': 1055,
    'adayinmylife': 921, 'morningroutine': 812, 'dayinmylife': 876,
    'viralvideo': 834, 'lifestylevlog': 833, 'reelsindia': 825,
    'india': 823, 'yt': 913, 'like': 834
}
bot15_lr_freq = {
    'studyroutine': 200, 'students': 220, 'shortsvideo': 250,
    'productive': 280, 'studytips': 290, 'asmr': 300,
    'minivlogs': 320, 'college': 330, 'trendingvideo': 350,
    'shortsviral': 360, 'foodie': 380, 'studyvlog': 390,
    'youtuber': 410, 'collegelife': 420, 'studentlife': 450
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('white')

for ax, freq, cmap, title, bg in [
    (axes[0], top15_lr_freq, 'Reds',  '✓  High Like-Rate Tags', '#BA3A2C'),
    (axes[1], bot15_lr_freq, 'Greys', '✗  Low Like-Rate Tags',  '#222222'),
]:
    wc = WordCloud(
        width=550, height=300,
        background_color='white',
        colormap=cmap,
        min_font_size=11, max_font_size=72,
        prefer_horizontal=0.85
    ).generate_from_frequencies(freq)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold',
                 color='white', backgroundcolor=bg, pad=8)

plt.tight_layout(pad=1.0)
plt.savefig('wc_template.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved wc_template.png")

# ── 2. Stacked Bar: Like Rate ──
all_types  = ['Identity', 'Content-specific', 'Audience-descriptor', 'Format-chasing', 'Ambiguous']
# Red family for bad types, dark for good types
type_colors = ['#222222', '#555555', '#BA3A2C', '#D97060', '#AAAAAA']

top_pcts = [27, 47, 7, 7, 13]
bot_pcts = [7,  13, 53, 27, 0]

fig, ax = plt.subplots(figsize=(10, 3.2))
fig.patch.set_facecolor('white')
bar_pos = [0.3, 1.1]
groups  = ['Top 15\n(High Like Rate)', 'Bottom 15\n(Low Like Rate)']

for i, (pos, pcts) in enumerate(zip(bar_pos, [top_pcts, bot_pcts])):
    left = 0
    for pct, color, ttype in zip(pcts, type_colors, all_types):
        if pct > 0:
            ax.barh(pos, pct, left=left, color=color,
                    label=ttype if i == 0 else '', height=0.45)
            if pct >= 10:
                ax.text(left + pct/2, pos, f'{pct}%',
                        ha='center', va='center', fontsize=9,
                        color='white', fontweight='bold')
            left += pct

ax.annotate('Identity 27%\nContent-specific 47%',
            xy=(37, 0.06), xytext=(37, -0.38),
            ha='center', fontsize=8.5, color='#222222', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=GRAY, lw=1.0))

ax.annotate('Audience-descriptor 53%\nFormat-chasing 27%',
            xy=(66, 1.44), xytext=(66, 1.78),
            ha='center', fontsize=8.5, color='#BA3A2C', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=GRAY, lw=1.0))

ax.set_yticks(bar_pos)
ax.set_yticklabels(groups, fontsize=11, color=BLACK)
ax.set_xlabel('Percentage (%)', fontsize=10, color=BLACK)
ax.set_xlim(0, 110)
ax.set_ylim(-0.75, 2.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', fontsize=8, ncol=2, framealpha=0.8)
ax.set_title('Tag Type Distribution: Top 15 vs Bottom 15 by Like Rate',
             fontsize=12, color=BLACK, fontweight='bold')
plt.tight_layout()
plt.savefig('bar_lr_template.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved bar_lr_template.png")

# ── 3. Stacked Bar: Views ──
type_colors_v = ['#222222', '#555555', '#BA3A2C', '#D97060', '#AAAAAA']
top_v = [13, 26, 13, 46, 0]
bot_v = [26, 40, 13, 0, 20]

fig, ax = plt.subplots(figsize=(10, 3.2))
fig.patch.set_facecolor('white')

for i, (pos, pcts) in enumerate(zip(bar_pos, [top_v, bot_v])):
    left = 0
    for pct, color, ttype in zip(pcts, type_colors_v, all_types):
        if pct > 0:
            ax.barh(pos, pct, left=left, color=color,
                    label=ttype if i == 0 else '', height=0.45)
            if pct >= 10:
                ax.text(left + pct/2, pos, f'{pct}%',
                        ha='center', va='center', fontsize=9,
                        color='white', fontweight='bold')
            left += pct

ax.annotate('Format-chasing 46%',
            xy=(55, 0.06), xytext=(55, -0.38),
            ha='center', fontsize=8.5, color='#BA3A2C', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=GRAY, lw=1.0))

ax.annotate('Identity 26%\nContent-specific 40%',
            xy=(33, 1.44), xytext=(33, 1.78),
            ha='center', fontsize=8.5, color='#222222', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=GRAY, lw=1.0))

ax.set_yticks(bar_pos)
ax.set_yticklabels(['Top 15\n(High Views)', 'Bottom 15\n(Low Views)'], fontsize=11, color=BLACK)
ax.set_xlabel('Percentage (%)', fontsize=10, color=BLACK)
ax.set_xlim(0, 110)
ax.set_ylim(-0.75, 2.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', fontsize=8, ncol=2, framealpha=0.8)
ax.set_title('Tag Type Distribution: Top 15 vs Bottom 15 by Views',
             fontsize=12, color=BLACK, fontweight='bold')
plt.tight_layout()
plt.savefig('bar_views_template.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved bar_views_template.png")

# ── 4. Hashtag Quantity Chart ──
labels = ['0', '1-3', '4-7', '8-15', '15+']
micro_lr    = [0.0254, 0.0154, 0.0139, 0.0129, 0.0144]
large_lr    = [0.0267, 0.0255, 0.0219, 0.0194, 0.0254]
micro_views = [259, 852, 930, 850, 791]
large_views = [1833736, 2218797, 3245543, 3664175, 5661214]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('white')

# Left: Views dual axis
ax1 = axes[0]
ax1r = ax1.twinx()
ax1.plot(labels, micro_views, marker='o', color=BLACK, linewidth=2,
         markersize=6, label='Micro (<10K)')
ax1r.plot(labels, [v/1e6 for v in large_views], marker='s', color=RED,
          linewidth=2, markersize=6, linestyle='--', label='Large (>10K)')
for i, v in enumerate(micro_views):
    ax1.text(i, v+18, str(v), ha='center', fontsize=7.5, color=BLACK)
for i, v in enumerate(large_views):
    ax1r.text(i, v/1e6+0.09, f'{v/1e6:.1f}M', ha='center', fontsize=7.5, color=RED)
ax1.set_ylim(0, 1400); ax1.set_yticks(range(0, 1401, 200))
ax1.set_ylabel('Median Views — Micro', color=BLACK, fontsize=9)
ax1r.set_ylabel('Median Views — Large (M)', color=RED, fontsize=9)
ax1.set_xlabel('Number of Hashtags', fontsize=10)
ax1.set_title('Discoverability Hypothesis', fontsize=11, fontweight='bold', color=BLACK)
ax1.spines['top'].set_visible(False)
lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax1r.get_legend_handles_labels()
ax1.legend(lines1+lines2, lab1+lab2, fontsize=8, loc='upper left')

# Right: Like Rate
axes[1].plot(labels, micro_lr, marker='o', color=BLACK, linewidth=2,
             markersize=6, label='Micro (<10K)')
axes[1].plot(labels, large_lr, marker='s', color=RED, linewidth=2,
             markersize=6, linestyle='--', label='Large (>10K)')
for i, (m, l) in enumerate(zip(micro_lr, large_lr)):
    axes[1].text(i, m+0.0005, f'{m:.3f}', ha='center', fontsize=7.5, color=BLACK)
    axes[1].text(i, l-0.0015, f'{l:.3f}', ha='center', fontsize=7.5, color=RED)
axes[1].set_xlabel('Number of Hashtags', fontsize=10)
axes[1].set_ylabel('Median Like Rate', fontsize=10)
axes[1].set_title('Audience Fit Hypothesis', fontsize=11, fontweight='bold', color=BLACK)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].legend(fontsize=8)

plt.suptitle('Does Hashtag Quantity Matter?', fontsize=13,
             fontweight='bold', color=BLACK)
plt.tight_layout()
plt.savefig('hashtag_template.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved hashtag_template.png")

# ── 5. Quadrant scatter ──
import pandas as pd
micro = pd.read_csv("micro_clean.csv")
micro = micro[(micro['view_count']>0)&(micro['subscriber_count']<10000)&(micro['like_rate']<=1)]
med_v = micro['view_count'].median()
med_l = micro['like_rate'].median()

def get_q(row):
    hv = row['view_count'] >= med_v
    hl = row['like_rate']  >= med_l
    if hv and hl:    return 'Golden'
    elif hv:         return 'Exposure Trap'
    elif hl:         return 'Niche'
    else:            return 'Ineffective'

micro['quadrant'] = micro.apply(get_q, axis=1)

qcolors = {'Golden': BLACK, 'Niche': '#BA3A2C', 'Exposure Trap': '#888888', 'Ineffective': '#CCCCCC'}

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('white')
for q, grp in micro.groupby('quadrant'):
    ax.scatter(grp['view_count'], grp['like_rate'],
               c=qcolors[q], label=q, alpha=0.5, s=18)
ax.axvline(med_v, color=GRAY, linestyle='--', linewidth=0.8)
ax.axhline(med_l, color=GRAY, linestyle='--', linewidth=0.8)
ax.set_xscale('log')
ax.set_xlabel('View Count (log scale)', fontsize=10, color=BLACK)
ax.set_ylabel('Like Rate', fontsize=10, color=BLACK)
ax.set_title('5.1 Tag Effectiveness Framework: Views × Like Rate',
             fontsize=11, fontweight='bold', color=BLACK)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('quadrant_template.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved quadrant_template.png")

Saved wc_template.png
Saved bar_lr_template.png
Saved bar_views_template.png
Saved hashtag_template.png
Saved quadrant_template.png
